# Lab: Fine-Tuning Pre-Trained Models

This notebook contains the starter code provided for the lab (with intentional errors for the debugging exercise).

Follow the lab steps to identify, analyze, and fix the errors, then fine-tune and evaluate the model.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from transformers import DataCollatorWithPadding

# Step 1: Load a Pre-Trained Model and Tokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
# The following line contains a typo in the parameter name ("num_labes" instead of "num_labels")
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# Step 2: Prepare the Fine-Tuning Dataset
# dataset = load_dataset("stanfordnlp/imdb", split="train[:2000]")
# def tokenize_function(examples):
#     return tokenizer(examples["text"], truncation=True)
# tokenized_datasets = dataset.map(tokenize_function, batched=True)
dataset = load_dataset("stanfordnlp/imdb", split="train[:2000]")
dataset = dataset.train_test_split(test_size=0.2)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)


# Step 3: Fine-Tune the Pre-Trained Model
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)
trainer.train()

# Step 4: Evaluate and Test the Fine-Tuned Model
test_input = "The movie was absolutely fantastic and I loved it."
test_encoding = tokenizer(test_input, return_tensors="pt")

# Move inputs to the same device as the model
device = next(model.parameters()).device
test_encoding = {k: v.to(device) for k, v in test_encoding.items()}

with torch.no_grad():
    logits = model(**test_encoding).logits

predicted_class = torch.argmax(logits, dim=1)
print("Predicted class:", predicted_class.item())
print("Class meaning:", "Positive" if predicted_class.item() == 1 else "Negative")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (13 outputs left out)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,No log,0.001300
2,No log,0.000599
3,No log,0.000478


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted class: 0
Class meaning: Negative


<b>Lab Report: Debugging and Fine-Tuning Process</b>

I began by reviewing the provided script and identified three intentional errors that would prevent successful execution. First, the model was loaded with a misspelled parameter (num_labes instead of num_labels). Second, num_train_epochs was supplied as a string ("3") rather than an integer. Third, the final inference call passed a dictionary to the model without unpacking it (model(test_encoding) instead of model(**test_encoding)). I also noticed several environment-specific issues that appeared once I began running the code on a newer version of the Hugging Face libraries.

I used an LLM to confirm and expand on these issues. I shared the original code and asked it to identify errors or misconfigurations. The responses confirmed the three intentional bugs and additionally flagged several version-related changes: the need to use the namespaced dataset path stanfordnlp/imdb, the rename of evaluation_strategy to eval_strategy, the removal of logging_dir, the rename of the tokenizer argument to processing_class in the Trainer, and the requirement to move test tensors to the same device as the model. These suggestions were precise and allowed me to make targeted edits rather than trial-and-error debugging.


After applying the fixes, I added a train/test split so evaluation could run, trained the DistilBERT model for three epochs on a 2,000-example subset of IMDB, and successfully obtained a prediction on a held-out test sentence. The process highlighted how effective targeted LLM prompting can be for locating both obvious syntax/API errors and subtler version-compatibility issues that arise when library interfaces change. The main remaining insight is that clear, section-specific prompts (rather than one large request) produced the most actionable feedback.


